# Vígil.ia — Detecção em vídeo: YOLO11-detect vs RT-DETR

**Objetivo:** sair da classificação (1 grão, foto) para **detecção** (vários grãos, vídeo).
Um detector acha **e** classifica cada grão num passo só — sem o recorte Otsu por fora.

**Experimento controlado** (mesmo dataset, mesmas épocas/imgsz): treina os dois e o **mAP decide**.
Mesma lógica do experimento EfficientNet vs YOLO que já fizemos.

Pré-requisito: dataset de **detecção** anotado (ver `PROTOCOLO_ANOTACAO_VIDEO.md`).
Classes (ordem canônica): `broken, immature, intact, skin-damaged, spotted`.

> Rodar no Colab com **GPU A100** (`Ambiente de execução → Alterar tipo → A100`).

## 1. Setup

In [ ]:
!pip -q install "ultralytics==8.4.80" roboflow

import torch, ultralytics
ultralytics.checks()
assert torch.cuda.is_available(), "Sem GPU! Troque o ambiente para A100."
print("GPU:", torch.cuda.get_device_name(0))

## 2. Dataset de detecção

Escolha **uma** das opções abaixo e ajuste. Ao final, `DATA_YAML` aponta para o `data.yaml` do export.

In [ ]:
import os, glob, yaml

# === Opção A: Roboflow API (recomendado) =====================================
USE_ROBOFLOW = True
ROBOFLOW_API_KEY = "COLE_SUA_KEY"
RF_WORKSPACE    = "seu-workspace"
RF_PROJECT      = "seu-projeto-deteccao"
RF_VERSION      = 1

# === Opção B: zip exportado no Google Drive ==================================
# USE_ROBOFLOW = False
# DRIVE_ZIP = "/content/drive/MyDrive/soja_deteccao.zip"  # export YOLOv11 do Roboflow

if USE_ROBOFLOW:
    from roboflow import Roboflow
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    ds = rf.workspace(RF_WORKSPACE).project(RF_PROJECT).version(RF_VERSION).download("yolov11")
    DATA_YAML = os.path.join(ds.location, "data.yaml")
else:
    from google.colab import drive
    drive.mount("/content/drive")
    !mkdir -p /content/soja_det && unzip -q -o "$DRIVE_ZIP" -d /content/soja_det
    hits = glob.glob("/content/soja_det/**/data.yaml", recursive=True)
    assert hits, "data.yaml não encontrado no zip"
    DATA_YAML = hits[0]

print("DATA_YAML:", DATA_YAML)
print(open(DATA_YAML).read())

In [ ]:
# Sanidade: classes na ordem certa + contagem de imagens/caixas por split
EXPECTED = ["broken", "immature", "intact", "skin-damaged", "spotted"]
cfg = yaml.safe_load(open(DATA_YAML))
names = cfg["names"] if isinstance(cfg["names"], list) else [cfg["names"][i] for i in sorted(cfg["names"])]
print("Classes do dataset:", names)
if [n.lower() for n in names] != EXPECTED:
    print("\n⚠️  ATENÇÃO: nomes/ordem diferentes do canônico", EXPECTED)
    print("   O front depende de 'intact' exatamente. Reanote/renomeie no Roboflow se preciso.")

base = os.path.dirname(DATA_YAML)
def _resolve(p):
    p = p.replace("../", "")
    cand = os.path.join(base, p)
    return cand if os.path.exists(cand) else os.path.join(base, os.path.basename(p))
for split in ["train", "val", "test"]:
    if split not in cfg: continue
    img_dir = _resolve(cfg[split])
    lbl_dir = img_dir.replace("images", "labels")
    imgs = glob.glob(os.path.join(img_dir, "*.*"))
    boxes = sum(len(open(f).read().split(chr(10))) for f in glob.glob(os.path.join(lbl_dir, "*.txt")))
    print(f"{split:6s}: {len(imgs):4d} imagens | ~{boxes} caixas")

## 3. Treino A — YOLO11-detect (CNN, baseline)

Leve, nativo do Ultralytics, treina rápido. `yolo11s` é um bom ponto de partida
(troque por `yolo11m` se tiver MUITO dado e quiser mais capacidade).

In [ ]:
from ultralytics import YOLO

EPOCHS = 100
IMGSZ  = 640
SEED   = 42

yolo = YOLO("yolo11s.pt")  # pesos COCO -> transfer learning
yolo.train(
    data=DATA_YAML, epochs=EPOCHS, imgsz=IMGSZ, batch=32, device=0, seed=SEED,
    patience=30, close_mosaic=10, optimizer="AdamW", lr0=0.001,
    project="runs_video", name="yolo11s_det", exist_ok=True,
    # aug extra p/ aproximar do domínio de vídeo (borrão/iluminação)
    hsv_v=0.5, degrees=15, translate=0.1, scale=0.5, fliplr=0.5, flipud=0.5, mosaic=1.0,
)

## 4. Treino B — RT-DETR (transformer)

Detector real-time baseado em DETR. Mais pesado e mais faminto por dado que a CNN —
por isso comparamos de forma justa, mesmas épocas/imgsz. `batch` menor (cabe na A100 40 GB).
Se der OOM, baixe o `batch`.

In [ ]:
from ultralytics import RTDETR

rtdetr = RTDETR("rtdetr-l.pt")  # pesos pré-treinados
rtdetr.train(
    data=DATA_YAML, epochs=EPOCHS, imgsz=IMGSZ, batch=16, device=0, seed=SEED,
    patience=30, close_mosaic=10, optimizer="AdamW", lr0=0.0001,
    project="runs_video", name="rtdetr_det", exist_ok=True,
    hsv_v=0.5, degrees=15, translate=0.1, scale=0.5, fliplr=0.5, flipud=0.5, mosaic=1.0,
)

## 5. Comparação no split de TESTE (o juiz)

Avalia os dois no mesmo `test`, mede mAP e velocidade. Quem ganhar vira o modelo de vídeo.

In [ ]:
def evaluate(weights, tag):
    m = YOLO(weights) if "yolo" in tag else RTDETR(weights)
    r = m.val(data=DATA_YAML, split="test", imgsz=IMGSZ, device=0, verbose=False)
    speed = sum(r.speed.values())  # ms por imagem (pre+inf+post)
    return {
        "tag": tag,
        "mAP50": round(r.box.map50, 4),
        "mAP50-95": round(r.box.map, 4),
        "ms/img": round(speed, 1),
        "fps": round(1000 / speed, 1) if speed else 0,
        "_r": r,
    }

res_yolo = evaluate("runs_video/yolo11s_det/weights/best.pt", "yolo11s")
res_rtd  = evaluate("runs_video/rtdetr_det/weights/best.pt",  "rtdetr-l")

print(f"{'modelo':10s} {'mAP50':>8s} {'mAP50-95':>10s} {'ms/img':>8s} {'fps':>7s}")
for r in (res_yolo, res_rtd):
    print(f"{r['tag']:10s} {r['mAP50']:>8.3f} {r['mAP50-95']:>10.3f} {r['ms/img']:>8.1f} {r['fps']:>7.1f}")

winner = max((res_yolo, res_rtd), key=lambda r: r["mAP50-95"])
print(f"\n🏆 Vencedor (mAP50-95): {winner['tag']}")

In [ ]:
# mAP por classe (onde cada modelo erra mais) — foco no 'intact' (regra Premium)
for res in (res_yolo, res_rtd):
    r = res["_r"]
    print(f"\n=== {res['tag']} — AP50-95 por classe ===")
    for i, ap in zip(r.box.ap_class_index, r.box.maps[r.box.ap_class_index]):
        print(f"  {names[i]:14s} {ap:.3f}")

## 6. Exportar o vencedor (.pt + ONNX)

`.pt` para servidor (HF Space) e `.onnx` caso a gente queira tentar no browser depois.

In [ ]:
import shutil
win_w = f"runs_video/{'yolo11s_det' if winner['tag']=='yolo11s' else 'rtdetr_det'}/weights/best.pt"
shutil.copy(win_w, "soja_detector_video.pt")

mdl = YOLO("soja_detector_video.pt") if winner["tag"] == "yolo11s" else RTDETR("soja_detector_video.pt")
mdl.export(format="onnx", imgsz=IMGSZ, opset=12, simplify=True)
print("Gerados: soja_detector_video.pt  +  .onnx")

# baixar pro PC (ou salvar no Drive)
from google.colab import files
files.download("soja_detector_video.pt")

## 7. Teste rápido em vídeo real

Roda o vencedor num vídeo de validação (grão na bandeja) e salva o resultado anotado.
É aqui que se confirma se o problema de vídeo foi resolvido.

In [ ]:
VIDEO_TESTE = "/content/drive/MyDrive/teste_esteira.mp4"  # ajuste
best = YOLO("soja_detector_video.pt") if winner["tag"] == "yolo11s" else RTDETR("soja_detector_video.pt")
best.predict(source=VIDEO_TESTE, conf=0.25, save=True, project="runs_video", name="pred_video", exist_ok=True)
print("Vídeo anotado salvo em runs_video/pred_video/")

# QC premium sobre as detecções de 1 frame (exemplo): % intacto >= 90% -> Aprovado
import numpy as np
r = best.predict(source=VIDEO_TESTE, conf=0.25, stream=True)
frame = next(iter(r))
cls_ids = frame.boxes.cls.cpu().numpy().astype(int)
total = len(cls_ids)
intact_id = names.index("intact")
premium = int((cls_ids == intact_id).sum())
pct = premium / total if total else 0
print(f"Frame: {total} grãos | {premium} intactos ({pct*100:.0f}%) -> {'APROVADO' if pct>=0.9 else 'REPROVADO'}")